Installing and importing the requirements

In [1]:
!pip install transformers datasets scikit-learn accelerate evaluate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [2]:
import re, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import warnings; warnings.filterwarnings('ignore')

In [3]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


Loading the GoEmotions Dataset provided by Google Research

**About the Dataset**

The **GoEmotions** dataset is a large-scale emotion classification dataset developed by Google Research, containing over 58,000 Reddit comments labeled with 27 fine-grained emotion categories plus a neutral class. Unlike traditional sentiment datasets that classify text as simply positive or negative, GoEmotions supports multi-label annotation, meaning a single comment can express multiple emotions at once (for example, both surprise and disappointment). It is widely used in natural language processing research for tasks such as emotion detection, affective computing, chatbot development, and social media analysis. The dataset is considered challenging due to overlapping emotions, informal Reddit language, and class imbalance, making it valuable for training and evaluating advanced transformer-based models like BERT and RoBERTa.


In [4]:
print("GoEmotions dataset")
raw = load_dataset('google-research-datasets/go_emotions', 'simplified')
print(raw)


GoEmotions dataset


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [5]:
EMOTION_LABELS = raw['train'].features['labels'].feature.names
NUM_LABELS = len(EMOTION_LABELS)
print(f'\nNumber of emotion classes: {NUM_LABELS}')
print(f'Labels: {EMOTION_LABELS}')


Number of emotion classes: 28
Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


Text Preprocessing for clean text

The function preprocesses raw text data by removing Reddit-specific mentions like u/username and r/subreddit, deleting URLs, stripping out all special characters and punctuation while keeping only letters, numbers, and spaces, then normalizing the text by collapsing multiple spaces into a single space, trimming extra whitespace, and converting everything to lowercase. It essentially converts messy social media text into a clean, standardized format suitable for NLP or machine learning tasks.

In [6]:
def clean_text(text):
    # Remove Reddit user/subreddit tokens
    text = re.sub(r'u/\w+|r/\w+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove special characters and punctuation, keep alphanumeric and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Collapse whitespace and lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

# Test
sample = raw['train'][0]
print(f"Original: {sample['text']}")
print(f"Cleaned:  {clean_text(sample['text'])}")


Original: My favourite food is anything I didn't have to cook myself.
Cleaned:  my favourite food is anything i didn t have to cook myself


Conversion to Multi-Label format

In [7]:
def make_multi_hot(labels_list, num_labels):
    vec = np.zeros(num_labels, dtype=np.float32)
    for l in labels_list:
        vec[l] = 1.0
    return vec

def preprocess(examples):
    texts = [clean_text(t) for t in examples['text']]
    labels = [make_multi_hot(l, NUM_LABELS) for l in examples['labels']]
    return {'text': texts, 'label_vector': labels}

processed = raw.map(preprocess, batched=True, remove_columns=['id', 'labels'])
print('Preprocessed dataset:')
print(processed)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Preprocessed dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 5427
    })
})


Stratified splits for train/test/val

 GoEmotions already has train/validation/test splits; we use them directly.
 For proper stratification on multi-label data, we use iterstrat.

In [8]:
train_data = processed['train']
val_data   = processed['validation']
test_data  = processed['test']

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')


Train: 43410 | Val: 5426 | Test: 5427


Analyzing the class distribution

In [9]:
train_labels = np.array(train_data['label_vector'])
label_counts = train_labels.sum(axis=0)
print('\nLabel distribution (top 5 most common):')
for i in np.argsort(label_counts)[::-1][:5]:
    print(f'  {EMOTION_LABELS[i]:20s}: {int(label_counts[i]):5d}')
print('\nLabel distribution (5 rarest):')
for i in np.argsort(label_counts)[:5]:
    print(f'  {EMOTION_LABELS[i]:20s}: {int(label_counts[i]):5d}')



Label distribution (top 5 most common):
  neutral             : 14219
  admiration          :  4130
  approval            :  2939
  gratitude           :  2662
  annoyance           :  2470

Label distribution (5 rarest):
  grief               :    77
  pride               :   111
  relief              :   153
  nervousness         :   164
  embarrassment       :   303


Tokenizer and the Dataset class

In [10]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128,
    )


tok_train = train_data.map(tokenize_fn, batched=True)
tok_val   = val_data.map(tokenize_fn, batched=True)
tok_test  = test_data.map(tokenize_fn, batched=True)

tok_train = tok_train.rename_column('label_vector', 'labels')
tok_val   = tok_val.rename_column('label_vector', 'labels')
tok_test  = tok_test.rename_column('label_vector', 'labels')


cols = ['input_ids', 'attention_mask', 'labels']
tok_train.set_format('torch', columns=cols)
tok_val.set_format('torch', columns=cols)
tok_test.set_format('torch', columns=cols)

print('Tokenization complete.')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Tokenization complete.


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
model.to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (torch.sigmoid(torch.tensor(logits)).numpy() > 0.5).astype(int)
    labels = labels.astype(int)
    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
    }

training_args = TrainingArguments(
    output_dir='./goemotions_distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print('Starting fine-tuning...')
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Model parameters: 66,975,004
Starting fine-tuning...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted
1,0.106803,0.100069,0.478431,0.203674,0.391689
2,0.089478,0.089176,0.537777,0.327415,0.482658
3,0.083117,0.088759,0.546729,0.340412,0.493017


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=4071, training_loss=0.12346733609103243, metrics={'train_runtime': 121.9115, 'train_samples_per_second': 1068.234, 'train_steps_per_second': 33.393, 'total_flos': 4314807064442880.0, 'train_loss': 0.12346733609103243, 'epoch': 3.0})

Evaluating on the test set

In [16]:
print('Evaluating on test set...')
test_results = trainer.evaluate(tok_test)
print('\nTest Results:')
for k, v in test_results.items():
    print(f'  {k}: {v:.4f}')

# Full classification report
preds_out = trainer.predict(tok_test)
preds = (torch.sigmoid(torch.tensor(preds_out.predictions)).numpy() > 0.5).astype(int)
true  = preds_out.label_ids.astype(int)

print('\nPer-class F1 scores:')
report = classification_report(true, preds, target_names=EMOTION_LABELS, zero_division=0, output_dict=True)
for emotion, metrics in sorted(report.items(), key=lambda x: x[1]['f1-score'] if isinstance(x[1], dict) else 0):
    if isinstance(metrics, dict) and emotion in EMOTION_LABELS:
        print(f'  {emotion:25s}: F1={metrics["f1-score"]:.3f}, support={metrics["support"]}')


Evaluating on test set...



Test Results:
  eval_loss: 0.0877
  eval_f1_micro: 0.5444
  eval_f1_macro: 0.3399
  eval_f1_weighted: 0.4936
  eval_runtime: 1.5149
  eval_samples_per_second: 3582.5210
  eval_steps_per_second: 56.1110
  epoch: 3.0000

Per-class F1 scores:
  disappointment           : F1=0.000, support=151.0
  embarrassment            : F1=0.000, support=37.0
  excitement               : F1=0.000, support=103.0
  grief                    : F1=0.000, support=6.0
  nervousness              : F1=0.000, support=23.0
  pride                    : F1=0.000, support=16.0
  realization              : F1=0.000, support=145.0
  relief                   : F1=0.000, support=11.0
  annoyance                : F1=0.077, support=320.0
  caring                   : F1=0.204, support=135.0
  disapproval              : F1=0.225, support=267.0
  desire                   : F1=0.255, support=83.0
  confusion                : F1=0.260, support=153.0
  disgust                  : F1=0.262, support=123.0
  approval              

In [17]:
def predict_emotions(texts, threshold=0.3):
    model.eval()
    cleaned = [clean_text(t) for t in texts]
    enc = tokenizer(cleaned, truncation=True, padding=True, max_length=128, return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.sigmoid(logits).cpu().numpy()
    results = []
    for i, prob in enumerate(probs):
        detected = [(EMOTION_LABELS[j], float(prob[j])) for j in np.where(prob > threshold)[0]]
        detected = sorted(detected, key=lambda x: -x[1])
        results.append({'text': texts[i], 'emotions': detected})
    return results

test_texts = [
    "I can't believe how amazing this is! I'm so thrilled!",
    "This is really frustrating and makes me angry.",
    "I feel so lonely and sad today.",
    "The results were surprising but I expected something different.",
]

preds_demo = predict_emotions(test_texts)

for r in preds_demo:
    print(f'\nText: "{r["text"][:80]}"')
    print(f'Emotions: {[(e, f"{p:.2f}") for e, p in r["emotions"]]}')


Text: "I can't believe how amazing this is! I'm so thrilled!"
Emotions: [('admiration', '0.58')]

Text: "This is really frustrating and makes me angry."
Emotions: [('anger', '0.64'), ('annoyance', '0.47')]

Text: "I feel so lonely and sad today."
Emotions: [('sadness', '0.79')]

Text: "The results were surprising but I expected something different."
Emotions: [('surprise', '0.73')]


In [22]:
print("""Strategy 1: Class-Weighted Loss
  → Compute inverse-frequency weights for rare classes.
  → Pass as pos_weight to BCEWithLogitsLoss inside a custom Trainer.
""")

N = len(train_labels)
pos_count = train_labels.sum(axis=0)
neg_count = N - pos_count
pos_weight = torch.tensor(neg_count / (pos_count + 1e-6), dtype=torch.float32).to(DEVICE)

print('Class weights (rare emotions get higher weight):')
rare_indices = np.argsort(pos_count)[:5]
for i in rare_indices:
    print(f'  {EMOTION_LABELS[i]:20s}: weight={pos_weight[i].item():.1f}, count={int(pos_count[i])}')

class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

weighted_trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight,
)

print("\nTraining with class-weighted loss...")
weighted_trainer.train()

weighted_results = weighted_trainer.evaluate(tok_test)

print(f"\nWeighted Loss Test F1 (micro): {weighted_results['eval_f1_micro']:.4f}")
print(f"Weighted Loss Test F1 (macro): {weighted_results['eval_f1_macro']:.4f}")

Strategy 1: Class-Weighted Loss
  → Compute inverse-frequency weights for rare classes.
  → Pass as pos_weight to BCEWithLogitsLoss inside a custom Trainer.

Class weights (rare emotions get higher weight):
  grief               : weight=562.8, count=77
  pride               : weight=390.1, count=111
  relief              : weight=282.7, count=153
  nervousness         : weight=263.7, count=164
  embarrassment       : weight=142.3, count=303

Training with class-weighted loss...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted
1,0.362302,1.031760,0.466713,0.421791,0.525745
2,0.352927,0.966270,0.442398,0.396766,0.512420
3,0.288753,1.016965,0.458882,0.407364,0.518337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Weighted Loss Test F1 (micro): 0.4581
Weighted Loss Test F1 (macro): 0.4046


In [23]:
print("""\nStrategy 2: Per-Class Threshold Tuning
  → Instead of a global 0.5 threshold, find the best threshold per class
    on the validation set to maximise per-class F1.
""")

val_preds = weighted_trainer.predict(tok_val)
val_logits = torch.sigmoid(torch.tensor(val_preds.predictions)).cpu().numpy()
val_true = val_preds.label_ids.astype(int)

thresholds = np.arange(0.1, 0.95, 0.05)
best_thresholds = np.zeros(NUM_LABELS)

for cls in range(NUM_LABELS):
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds_cls = (val_logits[:, cls] > t).astype(int)
        f1 = f1_score(val_true[:, cls], preds_cls, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[cls] = best_t

print("Per-class best thresholds (rare emotions often get lower threshold):")
for i in rare_indices:
    print(f"  {EMOTION_LABELS[i]:20s}: threshold={best_thresholds[i]:.2f}")

test_preds_out = weighted_trainer.predict(tok_test)
test_logits = torch.sigmoid(torch.tensor(test_preds_out.predictions)).cpu().numpy()
test_true = test_preds_out.label_ids.astype(int)

tuned_preds = np.zeros_like(test_logits, dtype=int)
for cls in range(NUM_LABELS):
    tuned_preds[:, cls] = (test_logits[:, cls] > best_thresholds[cls]).astype(int)

print(f"\nTuned Threshold Test F1 (micro):    {f1_score(test_true, tuned_preds, average='micro', zero_division=0):.4f}")
print(f"Tuned Threshold Test F1 (macro):    {f1_score(test_true, tuned_preds, average='macro', zero_division=0):.4f}")
print(f"Tuned Threshold Test F1 (weighted): {f1_score(test_true, tuned_preds, average='weighted', zero_division=0):.4f}")


Strategy 2: Per-Class Threshold Tuning
  → Instead of a global 0.5 threshold, find the best threshold per class
    on the validation set to maximise per-class F1.



Per-class best thresholds (rare emotions often get lower threshold):
  grief               : threshold=0.85
  pride               : threshold=0.80
  relief              : threshold=0.85
  nervousness         : threshold=0.85
  embarrassment       : threshold=0.90



Tuned Threshold Test F1 (micro):    0.5589
Tuned Threshold Test F1 (macro):    0.4774
Tuned Threshold Test F1 (weighted): 0.5687


In [27]:
import nltk
nltk.download('averaged_perceptron_tagger_eng', download_dir='/root/nltk_data')
nltk.download('wordnet', download_dir='/root/nltk_data')
nltk.download('omw-1.4', download_dir='/root/nltk_data')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [28]:
print("""\nStrategy 3: Data Augmentation for Rare Emotions
  → Augment rare-class samples using synonym replacement (nlpaug).
  → Increases effective training data for underrepresented emotions.
""")

try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "nlpaug", "-q"], check=True)
    import nlpaug.augmenter.word as naw

    aug = naw.SynonymAug(aug_src='wordnet', aug_max=3)

    augmented_texts, augmented_labels = [], []
    train_texts_list = train_data['text']
    train_labels_list = train_data['label_vector']

    for idx in range(len(train_texts_list)):
        label_vec = np.array(train_labels_list[idx])
        if any(label_vec[i] == 1 for i in rare_indices):
            original_text = train_texts_list[idx]
            augmented = aug.augment(original_text, n=2)
            for aug_text in augmented:
                augmented_texts.append(aug_text)
                augmented_labels.append(label_vec.tolist())

    print(f"Generated {len(augmented_texts)} augmented samples for rare emotions.")

    from datasets import Dataset
    aug_dataset = Dataset.from_dict({
        'text': augmented_texts,
        'label_vector': augmented_labels,
    })

    aug_tokenized = aug_dataset.map(tokenize_fn, batched=True)
    aug_tokenized = aug_tokenized.rename_column('label_vector', 'labels')
    aug_tokenized.set_format('torch', columns=cols)

    from torch.utils.data import ConcatDataset
    combined_train = ConcatDataset([tok_train, aug_tokenized])

    print(f"Combined training set size: {len(combined_train)}")
    print("→ Re-train WeightedTrainer on combined_train for best rare-emotion performance.")

except Exception as e:
    print(f"nlpaug augmentation skipped: {e}")
    print("Install with: pip install nlpaug")


Strategy 3: Data Augmentation for Rare Emotions
  → Augment rare-class samples using synonym replacement (nlpaug).
  → Increases effective training data for underrepresented emotions.

Generated 1606 augmented samples for rare emotions.


Map:   0%|          | 0/1606 [00:00<?, ? examples/s]

Combined training set size: 45016
→ Re-train WeightedTrainer on combined_train for best rare-emotion performance.


In [30]:
from transformers import AutoModelForSequenceClassification

model_aug = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
).to(DEVICE)

weighted_trainer_aug = WeightedTrainer(
    model=model_aug,
    args=training_args,
    train_dataset=combined_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight,
)

print("Training WeightedTrainer with augmented data...")
weighted_trainer_aug.train()

aug_results = weighted_trainer_aug.evaluate(tok_test)

print(f"\nAugmented + Weighted Test F1 (micro): {aug_results['eval_f1_micro']:.4f}")
print(f"Augmented + Weighted Test F1 (macro): {aug_results['eval_f1_macro']:.4f}")
print(f"Augmented + Weighted Test F1 (weighted): {aug_results['eval_f1_weighted']:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training WeightedTrainer with augmented data...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted
1,0.812915,0.713108,0.342507,0.301006,0.450103
2,0.627164,0.641851,0.370447,0.325795,0.475405
3,0.531074,0.658539,0.395901,0.345028,0.488468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Augmented + Weighted Test F1 (micro): 0.3911
Augmented + Weighted Test F1 (macro): 0.3370
Augmented + Weighted Test F1 (weighted): 0.4843


In [29]:
print("""\nStrategy 4: Focal Loss
  → Down-weights easy (well-classified) examples and focuses training
    on hard, rare examples. gamma controls the focusing strength.
""")

class FocalLossTrainer(Trainer):
    def __init__(self, *args, gamma=2.0, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits,
            labels,
            pos_weight=self.pos_weight,
            reduction='none'
        )

        probs = torch.sigmoid(logits)
        p_t = labels * probs + (1 - labels) * (1 - probs)
        focal_w = (1 - p_t) ** self.gamma
        loss = (focal_w * bce).mean()

        return (loss, outputs) if return_outputs else loss

focal_trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    gamma=2.0,
    pos_weight=pos_weight,
)

print("Training with Focal Loss (gamma=2.0) + class weights...")
focal_trainer.train()

focal_results = focal_trainer.evaluate(tok_test)

print(f"\nFocal Loss Test F1 (micro): {focal_results['eval_f1_micro']:.4f}")
print(f"Focal Loss Test F1 (macro): {focal_results['eval_f1_macro']:.4f}")


Strategy 4: Focal Loss
  → Down-weights easy (well-classified) examples and focuses training
    on hard, rare examples. gamma controls the focusing strength.

Training with Focal Loss (gamma=2.0) + class weights...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted
1,0.054643,0.708452,0.498926,0.445968,0.535664
2,0.086238,0.378592,0.455840,0.410703,0.516525
3,0.068253,0.448699,0.472397,0.422030,0.521059


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Focal Loss Test F1 (micro): 0.4938
Focal Loss Test F1 (macro): 0.4366
